# Camada Gold — Modelagem Dimensional (Esquema Estrela)

Esta camada transforma os dados limpos da Silver em um **modelo dimensional** (esquema estrela), otimizado para responder as perguntas de negócio do MVP.

**Estrutura:**
- `fato_vendas` — tabela fato no grão de **item de pedido**, com as métricas: preço, frete, prazo de entrega, atraso e nota da avaliação.
- `dim_produto`, `dim_cliente`, `dim_vendedor`, `dim_tempo` — dimensões que descrevem e contextualizam cada venda.

## fato_vendas — construção

Grão: um item de pedido. Construído a partir de `silver.order_items`, enriquecido com dados de `orders` (datas, cliente, status) e `order_reviews` (nota).

In [0]:
# Camada Gold — construção do fato_vendas (grão: item de pedido).
# Carga das tabelas Silver que compõem o fato.
df_items = spark.table("ecommerce_br.silver.order_items")
df_orders = spark.table("ecommerce_br.silver.orders")
df_reviews = spark.table("ecommerce_br.silver.order_reviews")

print("Tabelas Silver carregadas.")
print(f"order_items: {df_items.count()}")
print(f"orders: {df_orders.count()}")
print(f"order_reviews: {df_reviews.count()}")

Tabelas Silver carregadas.
order_items: 112650
orders: 99441
order_reviews: 99224


In [0]:
# Etapa A: junção de order_items com orders pelo order_id.
# Traz para cada item as datas do pedido, o cliente e o status.
# Left join a partir de order_items: preserva todos os itens (o fato parte do item).
df_fato = df_items.join(
    df_orders.select(
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ),
    on="order_id",
    how="left"
)

print(f"Linhas após junção com orders: {df_fato.count()}")

Linhas após junção com orders: 112650


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

# Deduplicação de avaliações: manter uma por pedido (a mais recente).
# 547 pedidos têm múltiplas avaliações, o que multiplicaria itens no join.
# A window numera as avaliações de cada pedido da mais recente para a mais antiga;
# mantém-se apenas a primeira (row_number = 1), preservando a opinião final do cliente.
janela = Window.partitionBy("order_id").orderBy(col("review_creation_date").desc())

df_reviews_dedup = df_reviews \
    .withColumn("rn", row_number().over(janela)) \
    .filter(col("rn") == 1) \
    .select("order_id", "review_score")

print(f"Avaliações originais: {df_reviews.count()}")
print(f"Avaliações após dedup (1 por pedido): {df_reviews_dedup.count()}")

Avaliações originais: 99224
Avaliações após dedup (1 por pedido): 98673


In [0]:
# Etapa B (corrigida): junção com as avaliações deduplicadas (1 por pedido).
# Uso de df_reviews_dedup evita a multiplicação de itens causada por pedidos
# com múltiplas avaliações (547 pedidos, detectados na conferência de contagem).
df_fato = df_fato.join(
    df_reviews_dedup,
    on="order_id",
    how="left"
)

print(f"Linhas após junção com reviews (corrigida): {df_fato.count()}")

Linhas após junção com reviews (corrigida): 112650


### Verificação de qualidade — avaliações duplicadas

Conferência que motivou a deduplicação: alguns pedidos possuem mais de uma avaliação, o que multiplicaria os itens no join. Detectados 547 pedidos nessa condição, tratados mantendo a avaliação mais recente.

In [0]:
# Investigação: pedidos com mais de uma avaliação (causa da multiplicação no join).
from pyspark.sql.functions import col, count

pedidos_multi_review = df_reviews.groupBy("order_id").agg(count("*").alias("qtd_reviews")) \
    .filter(col("qtd_reviews") > 1)

print(f"Pedidos com mais de 1 avaliação: {pedidos_multi_review.count()}")
pedidos_multi_review.orderBy(col("qtd_reviews").desc()).show(5, truncate=False)

Pedidos com mais de 1 avaliação: 547
+--------------------------------+-----------+
|order_id                        |qtd_reviews|
+--------------------------------+-----------+
|8e17072ec97ce29f0e1f111e598b0c85|3          |
|03c939fd7fd3b38f8485a0f95798f1f6|3          |
|df56136b8031ecd28e200bb18e6ddb2e|3          |
|c88b1d1b157a9999ce368f218a407141|3          |
|60b3fa9424e97cd3b6b95ca799d511c6|2          |
+--------------------------------+-----------+
only showing top 5 rows


In [0]:
from pyspark.sql.functions import datediff

# Etapa C: cálculo das métricas temporais de entrega.
# prazo_entrega_dias: dias entre a compra e a entrega ao cliente (tempo total de entrega).
# atraso_dias: dias entre a entrega real e a data prevista.
#   > 0 = entregue com atraso; < 0 = entregue adiantado; 0 = no prazo.
# Ambas resultam em NULL para pedidos não entregues (data de entrega nula) — comportamento
# correto: só há prazo/atraso para pedidos efetivamente entregues.
df_fato = df_fato \
    .withColumn("prazo_entrega_dias",
                datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp"))) \
    .withColumn("atraso_dias",
                datediff(col("order_delivered_customer_date"), col("order_estimated_delivery_date")))

# Conferência: amostra das métricas para pedidos entregues.
df_fato.select(
    "order_id",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "prazo_entrega_dias",
    "atraso_dias"
).filter(col("order_delivered_customer_date").isNotNull()).show(5, truncate=False)

+--------------------------------+------------------------+-----------------------------+-----------------------------+------------------+-----------+
|order_id                        |order_purchase_timestamp|order_delivered_customer_date|order_estimated_delivery_date|prazo_entrega_dias|atraso_dias|
+--------------------------------+------------------------+-----------------------------+-----------------------------+------------------+-----------+
|f271576bed568e896f99eb710cd3a6f8|2018-01-07 21:44:54     |2018-01-17 20:26:31          |2018-02-14 00:00:00          |10                |-28        |
|3bc77ce8be27211bac313c2daa402d1a|2017-04-06 22:39:29     |2017-04-11 12:31:36          |2017-04-27 00:00:00          |5                 |-16        |
|e4de6d53ecff736bc68804b0b6e9f635|2017-10-16 14:56:50     |2017-11-08 21:25:24          |2017-11-21 00:00:00          |23                |-13        |
|66d03fb3c4b0f80b5db04ef9c260336d|2018-02-06 21:09:10     |2018-02-16 20:24:42          |2018-

In [0]:
# Etapa D: seleção das colunas finais do fato e persistência.
# O fato reúne as chaves (para ligar às dimensões) e as métricas de negócio.
# Datas cruas usadas apenas para derivar prazo/atraso não são mantidas no fato final.
from pyspark.sql.functions import current_timestamp

df_fato_vendas = df_fato.select(
    # Chaves de ligação com as dimensões
    "order_id",
    "order_item_id",
    "product_id",
    "seller_id",
    "customer_id",
    # Contexto do pedido
    "order_status",
    "order_purchase_timestamp",
    # Métricas de valor
    "price",
    "freight_value",
    # Métricas de entrega (calculadas na Etapa C)
    "prazo_entrega_dias",
    "atraso_dias",
    # Métrica de satisfação
    "review_score"
).withColumn("data_processamento_gold", current_timestamp())

# Conferência final antes de gravar.
df_fato_vendas.printSchema()
print(f"Total de linhas no fato_vendas: {df_fato_vendas.count()}")

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- price: decimal(10,2) (nullable = true)
 |-- freight_value: decimal(10,2) (nullable = true)
 |-- prazo_entrega_dias: integer (nullable = true)
 |-- atraso_dias: integer (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- data_processamento_gold: timestamp (nullable = false)

Total de linhas no fato_vendas: 112650


In [0]:
# Persistência do fato_vendas no schema gold.
df_fato_vendas.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ecommerce_br.gold.fato_vendas")

print("Tabela ecommerce_br.gold.fato_vendas criada com sucesso!")

Tabela ecommerce_br.gold.fato_vendas criada com sucesso!


## dim_produto — dimensão de produtos

Dimensão descritiva dos produtos, com a categoria (em português e inglês) e atributos físicos. Chave: `product_id`.

In [0]:
# Dimensão de produtos: atributos descritivos para análise por produto/categoria.
# Parte da silver.products (já limpa e tipada). Chave: product_id.
df_dim_produto = spark.table("ecommerce_br.silver.products").select(
    "product_id",
    "product_category_name",
    "product_category_name_english",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
).withColumn("data_processamento_gold", current_timestamp())

df_dim_produto.printSchema()
print(f"Total de produtos (dim_produto): {df_dim_produto.count()}")

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)
 |-- data_processamento_gold: timestamp (nullable = false)

Total de produtos (dim_produto): 32951


In [0]:
# Persistência da dimensão de produtos no schema gold.
df_dim_produto.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ecommerce_br.gold.dim_produto")

print("Tabela ecommerce_br.gold.dim_produto criada com sucesso!")

Tabela ecommerce_br.gold.dim_produto criada com sucesso!


## dim_vendedor — dimensão de vendedores

Dimensão descritiva dos vendedores, com localização (cidade, estado, CEP) e coordenadas geográficas (via join com geolocation pelo CEP). As coordenadas habilitam o cálculo de distância cliente×vendedor (P5). Chave: `seller_id`.

In [0]:
# Dimensão de vendedores: localização + coordenadas geográficas.
# Coordenadas obtidas por left join com a geolocation agregada (1 ponto por CEP),
# habilitando o cálculo de distância cliente x vendedor (P5).
# Left join preserva todos os vendedores mesmo se o CEP não tiver coordenada.
df_sellers = spark.table("ecommerce_br.silver.sellers")
df_geo = spark.table("ecommerce_br.silver.geolocation")

df_dim_vendedor = df_sellers.join(
    df_geo.select(
        col("geolocation_zip_code_prefix"),
        col("geolocation_lat").alias("seller_lat"),
        col("geolocation_lng").alias("seller_lng")
    ),
    df_sellers.seller_zip_code_prefix == df_geo.geolocation_zip_code_prefix,
    how="left"
).select(
    "seller_id",
    "seller_zip_code_prefix",
    "seller_city",
    "seller_state",
    "seller_lat",
    "seller_lng"
).withColumn("data_processamento_gold", current_timestamp())

df_dim_vendedor.printSchema()
print(f"Total de vendedores (dim_vendedor): {df_dim_vendedor.count()}")

root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: string (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)
 |-- seller_lat: double (nullable = true)
 |-- seller_lng: double (nullable = true)
 |-- data_processamento_gold: timestamp (nullable = false)

Total de vendedores (dim_vendedor): 3095


In [0]:
# Verificação: vendedores sem coordenada (CEP ausente na geolocation).
# Relevante para a P5 — sem coordenada não há cálculo de distância.
sem_coord = df_dim_vendedor.filter(col("seller_lat").isNull()).count()
print(f"Vendedores sem coordenada: {sem_coord} de 3095")

Vendedores sem coordenada: 7 de 3095


In [0]:
# Persistência da dimensão de vendedores no schema gold.
df_dim_vendedor.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ecommerce_br.gold.dim_vendedor")

print("Tabela ecommerce_br.gold.dim_vendedor criada com sucesso!")

Tabela ecommerce_br.gold.dim_vendedor criada com sucesso!


## dim_cliente — dimensão de clientes

Dimensão descritiva dos clientes, com localização (cidade, estado, CEP) e coordenadas geográficas (via join com geolocation pelo CEP). Mantém os dois identificadores do cliente. As coordenadas habilitam o cálculo de distância cliente×vendedor (P5). Chave: `customer_id`.

In [0]:
# Dimensão de clientes: localização + coordenadas geográficas.
# Mesma lógica da dim_vendedor: left join com a geolocation pelo CEP.
# Mantém customer_id (liga ao fato) e customer_unique_id (identifica a pessoa real).
df_customers = spark.table("ecommerce_br.silver.customers")

df_dim_cliente = df_customers.join(
    df_geo.select(
        col("geolocation_zip_code_prefix"),
        col("geolocation_lat").alias("customer_lat"),
        col("geolocation_lng").alias("customer_lng")
    ),
    df_customers.customer_zip_code_prefix == df_geo.geolocation_zip_code_prefix,
    how="left"
).select(
    "customer_id",
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state",
    "customer_lat",
    "customer_lng"
).withColumn("data_processamento_gold", current_timestamp())

df_dim_cliente.printSchema()
print(f"Total de clientes (dim_cliente): {df_dim_cliente.count()}")

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- customer_lat: double (nullable = true)
 |-- customer_lng: double (nullable = true)
 |-- data_processamento_gold: timestamp (nullable = false)

Total de clientes (dim_cliente): 99441


In [0]:
# Verificação: clientes sem coordenada (CEP ausente na geolocation).
sem_coord_cli = df_dim_cliente.filter(col("customer_lat").isNull()).count()
print(f"Clientes sem coordenada: {sem_coord_cli} de 99441")

Clientes sem coordenada: 278 de 99441


In [0]:
# Persistência da dimensão de clientes no schema gold.
df_dim_cliente.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ecommerce_br.gold.dim_cliente")

print("Tabela ecommerce_br.gold.dim_cliente criada com sucesso!")

Tabela ecommerce_br.gold.dim_cliente criada com sucesso!


## dim_tempo — dimensão de tempo

Dimensão temporal derivada das datas de compra, desdobrada em componentes (ano, mês, dia, trimestre, nome do mês, dia da semana) para habilitar análises de sazonalidade. Chave: `data_compra`.

In [0]:
from pyspark.sql.functions import to_date, year, month, dayofmonth, quarter, date_format, dayofweek

# Dimensão de tempo: uma linha por data de compra distinta, com componentes derivados.
# Construída a partir das datas de compra do fato, permitindo análises temporais
# (sazonalidade, tendências) — não exigidas pelas 5 perguntas, mas incluídas para
# completar o modelo dimensional e habilitar trabalhos futuros.
df_datas = spark.table("ecommerce_br.gold.fato_vendas") \
    .select(to_date(col("order_purchase_timestamp")).alias("data_compra")) \
    .distinct()

df_dim_tempo = df_datas.select(
    "data_compra",
    year("data_compra").alias("ano"),
    month("data_compra").alias("mes"),
    dayofmonth("data_compra").alias("dia"),
    quarter("data_compra").alias("trimestre"),
    date_format("data_compra", "MMMM").alias("nome_mes"),
    dayofweek("data_compra").alias("dia_semana")
).withColumn("data_processamento_gold", current_timestamp())

df_dim_tempo.printSchema()
df_dim_tempo.orderBy("data_compra").show(5, truncate=False)
print(f"Total de datas (dim_tempo): {df_dim_tempo.count()}")

root
 |-- data_compra: date (nullable = true)
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- dia: integer (nullable = true)
 |-- trimestre: integer (nullable = true)
 |-- nome_mes: string (nullable = true)
 |-- dia_semana: integer (nullable = true)
 |-- data_processamento_gold: timestamp (nullable = false)

+-----------+----+---+---+---------+---------+----------+--------------------------+
|data_compra|ano |mes|dia|trimestre|nome_mes |dia_semana|data_processamento_gold   |
+-----------+----+---+---+---------+---------+----------+--------------------------+
|2016-09-04 |2016|9  |4  |3        |September|1         |2026-09-18 02:15:42.362651|
|2016-09-05 |2016|9  |5  |3        |September|2         |2026-09-18 02:15:42.362651|
|2016-09-15 |2016|9  |15 |3        |September|5         |2026-09-18 02:15:42.362651|
|2016-10-02 |2016|10 |2  |4        |October  |1         |2026-09-18 02:15:42.362651|
|2016-10-03 |2016|10 |3  |4        |October  |2         |2026-09-1

In [0]:
# Persistência da dimensão de tempo no schema gold.
df_dim_tempo.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ecommerce_br.gold.dim_tempo")

print("Tabela ecommerce_br.gold.dim_tempo criada com sucesso!")

Tabela ecommerce_br.gold.dim_tempo criada com sucesso!
